# A Toy Example

See `prep_toy/generate_toy.ipynb` for the generation of toy data.

## Set Up

In [ ]:
from pathlib import Path
import ace_of_clust as aoc

base_dir =  Path("..").resolve()
example_data_dir = base_dir / "examples" / "data" / "toy"


## Run *Clumppling* alignment for each model
Models here are different clustering algorithms:
- NMF: Non-negative Matrix Factorization (mixed-membership clustering)
- LDA: Latent Dirichlet Allocation (mixed-membership clustering)
- K-Means: K-Means Clustering (hard clustering)

In [ ]:
for method_lb in ["nmf", "lda", "kmeans"]:

    cls_dir = example_data_dir / "clustering" / method_lb
    align_dir = example_data_dir / "aligned" / method_lb

    # --- call clumppling via the wrapper ----------------------------------------
    aoc.run_clumppling_via_main(
        input_dir=cls_dir,
        output_dir=align_dir,
        fmt="generalQ",                    # -f generalQ
        extension=".Q",
        vis=False,                         # -v F
        use_rep=True,                      # --use_rep T
        use_best_pair=True,                # --use_best_pair T
        merge=True,                        # --merge T
        cd_res=1.0,                        # --cd_res 1.0
        comm_max=0.1,
    )


## Run *Clumppling.compModels*

Prepare *compModels* inputs.

In [ ]:
model_comp_dir = example_data_dir / "comp_models" / "toy"
model_comp_output_dir = example_data_dir / "comp_models" / "toy_output"
models = ["nmf", "lda", "kmeans"]
suffixes = ["rep"] * len(models)

model_dirs = [
    example_data_dir / "aligned" / "nmf",
    example_data_dir / "aligned" / "lda",
    example_data_dir / "aligned" / "kmeans",
]

# Prepare qfilelist / qnamelist / mode_stats files
qfilelists, qnamelists, mode_stats_files = aoc.prepare_comp_models_inputs(
    models=models,
    model_dirs=model_dirs,
    comp_dir=model_comp_dir,
    suffixes=suffixes,
)


[Optional] Add ground truth as an additional "model" results.

In [ ]:
gt_qfile = example_data_dir / "ground_truth.Q"
gt_qfile_file = example_data_dir / "comp_models" / f"toy" / "truth.qfilelist"
# write this to a .qfilelist file
with open(gt_qfile_file, "w") as f:
    f.write(str(gt_qfile) + "\n")
gt_qname_file = example_data_dir / "comp_models" / f"toy" / "truth.qnamelist"
# write this to a .qnamelist file
with open(gt_qname_file, "w") as f:
    f.write("ground.truth\n")
qfilelists.insert(0, str(gt_qfile_file))
qnamelists.insert(0, str(gt_qname_file))
# add "ground.truth" mode_stats file
gt_mode_stats_file = model_comp_dir / "dummy" / "mode_stats.txt"
# make parent dir if not exists
gt_mode_stats_file.parent.mkdir(parents=True, exist_ok=True)
with open(gt_mode_stats_file, "w") as f:
    f.write('Mode,Representative,Size,Cost,Performance\n')
    f.write("K4M1,ground.truth,1,0.0,1.0\n")
mode_stats_files.insert(0, gt_mode_stats_file)
# add "ground.truth" to models list
models.insert(0, "ground.truth")

Run *compModels*.

In [ ]:
aoc.run_comp_models(
    models=models,
    comp_dir=model_comp_dir,
    output_dir=model_comp_output_dir,
    vis=False,
    bg_colors=None,   
    include_sim_in_label=True,
    ind_labels="",    
    qfilelists=qfilelists,
    qnamelists=qnamelists,
    mode_stats_files=mode_stats_files,
)

## Analyze *ACE-OF-Clust* Results

### Preparation

In [ ]:
from importlib.resources import files
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

#### load colors
Colors are customizable and should be provided as a list of color codes (or named colors). Here, we use the package’s built-in default colors.

In [ ]:
path = files("ace_of_clust").joinpath("resources/default_colors.txt")
default_colors = path.read_text().splitlines()
fig, _, _  = aoc.plot_discrete_colorbar(default_colors[:5], K_max=5)

#### load truth labels
If annotations are available (e.g., ground truth, manual annotations, or reference labels), we load them here to be used in downstream label-based analyses and visualizations.

In [ ]:
truth_labels_file = example_data_dir / "clustering" / "kmeans" / "groups.txt"
truth_labels = np.loadtxt(truth_labels_file, dtype=int)
print(pd.Series(truth_labels).value_counts())

#### load truth X and compute coordinates (via PCA)
We load the clustering input data $X$ if available (e.g., a gene-expression count matrix).

Because our toy dataset has no associated **coordinates**, we compute PCA (via sklearn) and use PC1 and PC2 as coordinates.

For scRNA-seq data, coordinates are typically UMAP embeddings; for spatial transcriptomics, they should be the spatial coordinates. 

The notion of “coordinates” is flexible—it can even be the values of two selected features—and is mainly used to support 2D visualization.

In [ ]:
# load truth X and compute PCA (using sklearn) to be used as coordinates
truth_X_file = example_data_dir / "X.txt"
truth_X = np.loadtxt(truth_X_file, delimiter=",")
X_pca = PCA(n_components=2).fit_transform(truth_X)
feature_names = [f"Feature{i+1}" for i in range(truth_X.shape[1])]

#### visualize ground truth labels

In [ ]:
plt.figure(figsize=(4,4), dpi=150)
plt.scatter(X_pca[:,0], X_pca[:,1], c=truth_labels, cmap='tab10', s=10, alpha=0.8)
# label each group in the center
for label in np.unique(truth_labels):
    mask = truth_labels == label
    x_center = X_pca[mask, 0].mean()
    y_center = X_pca[mask, 1].mean()
    plt.text(x_center, y_center, str(label), color='black', fontsize=12, ha='center', va='center', weight='bold')
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Ground Truth")
plt.show()

#### visualize (some selected) feature value 

This is used to visualize gene counts, etc.

In [ ]:
i_g = 0 # first feature
feature_val = truth_X[:, i_g]
fig, ax = plt.subplots(1, 1, figsize=(3,3), dpi=150)
fig = aoc.plot_feature_count(feature_val, X_pca, feature_name=feature_names[i_g], ax=ax,
                             cmap="RdYlBu_r", cbar_label="Value")

### Model 1: KMeans

### Model 2: LDA

In [ ]:
align_dir = example_data_dir / "aligned" / "lda"
cls_dir = example_data_dir / "clustering" / "lda"

#### load results

In [ ]:
# load clumppling results (with P matrices)
results = aoc.load_clumppling_results(
    align_dir=align_dir,
    suffix="rep",
    cls_dir=cls_dir,
    load_P=True,  # set to True to load P (for mixed-membership models)
    strict_P=True,   # raises FileNotFoundError if any P file is missing
)

# compute pairwise mappings between modes
pair_mappings = aoc.extract_all_mode_pair_mappings(
    mode_names=results.modes,
    all_modes_alignment=results.all_modes_alignment,
    alignment_acrossK=results.alignment_acrossK,
)

# compute per-gene metrics for all modes
df_by_mode = aoc.compute_feature_metrics_all_modes(results, feature_names=feature_names)

# select top features by weighted_Psum quantile across all modes 
selected_by_mode, df_selected_all, overlap = aoc.select_top_features_by_weighted_Psum(
    df_by_mode,
    top_quantile=0.1,
)


### Comparison of Multiple Models

In [ ]:
# Load the comparison results
comp_res = aoc.load_compmodels_results(
    res_dir=model_comp_output_dir,
    input_dir=model_comp_dir,
)

# Extract mode-pair mappings across all models
pair_mappings = aoc.extract_all_mode_pair_mappings(
    mode_names=comp_res.full_mode_names,
    all_modes_alignment=comp_res.all_modes_alignment,
    alignment_acrossK=comp_res.alignment_across_all,
)

# Display names of all modes
print(f"Loaded {len(comp_res.full_mode_names)} aligned modes from {len(comp_res.models)} models.")
models = comp_res.models
for model in models:
    print(f" Model: {model}")
    print("\t", comp_res.modes_by_model[model])

#### plot aligned clusteirng results
Here, we plot cluster memberships on a 2D scatter plot (here using PCA coordinates), with colors indicating clusters.

By setting `val_threshold`, we only display/color points whose membership exceeds the threshold. For hard clustering, any value in (0,1) works (typically $>10^{−6}$); for mixed-membership clustering, this threshold controls plot density and overlap and should be chosen accordingly. We recommend $0.1–0.5$.

In [ ]:
# all modes
fig, ax = aoc.plot_compmodels_membership_grid(
    comp_res,
    X_pca, # coordinates for scatter plot
    colors=default_colors[:comp_res.K_max],
    val_threshold=0.1, # only plot points with membership values above this threshold
    suptitle="Cluster Memberships",
    y_suptitle=0.91,
    s=5, 
    models_plot_order=models,
)

We can also highlight points whose clustering differs substantially from a reference mode (specified by `ref_mode`). 

With `diff_threshold`, we display only points whose membership differences exceed this value.

In [ ]:
ref_mode = "ground.truth_ground.truth"
print(f"Comparing to {ref_mode}")

fig_diff, axes_diff = aoc.plot_compmodels_diff_grid_against_ref(
    comp_res=comp_res,
    pair_mappings=pair_mappings,
    coords=X_pca,
    ref_mode=ref_mode, # the full name of the reference mode to compare to
    models_plot_order=models[1:],  # exclude ground truth from the diff plots
    diff_threshold=0.1, # only plot points with differences above this threshold
    val_threshold=0.1, # only plot points with reference membership values above this threshold
    colors=default_colors[:comp_res.K_max],
    s=5, alpha=0.9,  # point size and transparency
    suptitle="Difference in Cluster Memberships",
    y_suptitle=0.92,
)

Instead of all modes, we can also visualize a subset of selected modes. Here, we choose to plot all major modes (the one with largest number of runs in each model).

In [ ]:
# get the major mode (with largest size) from each model
modes_to_plot = list()
for model in models[1:]:  # skip ground truth
    major_mode = comp_res.mode_stats_by_model[model].sort_values(by='Size', ascending=False).index.values[0]
    modes_to_plot.append((model, major_mode))

fig, axes = aoc.plot_compmodels_membership_selected(
    comp_res,
    X_pca,
    model_mode_list=modes_to_plot,
    n_rows=1,          # number of rows in the plot grid
    colors=default_colors,  
    suptitle="Cluster Memberships of Selected Modes",
    y_suptitle=1.08,
    figsize_scale=(2.5,1.8),
    s=5,
)
# update individual titles
for model, mode in modes_to_plot:
    axes[(model, mode)].set_title(model.upper(), fontsize=9, weight='bold', loc='left')

We use the alignment pattern graph to track how clusters are aligned across modes and models.

In [ ]:
fig, ax = aoc.plot_compmodels_alignment_by_model(
    comp_res,
    cmap=default_colors[:comp_res.K_max],
    pair_mappings=pair_mappings,
    connect_identity=False,  # only highlight non-1–1 / shifted alignments
    adjacent_only=True,      # only between neighboring model columns
    label_modes=True,   # show mode names on the corner
    alt_ls=True, ls_alt=("-", "--", ":", "-."), lw=0.6, # line styles for different K
    figsize_scale=(0.3, 3),
    wspace_padding=1.5,
    dpi=300,
    row_by_K=True, # put modes with same K in the same row
    models_plot_order=models,
)

# update y labels and titles
ax.set_yticklabels(['K=3', 'K=4', 'K=5', '', '', '', ''], fontsize=11)
ax.set_ylabel("Modes", fontsize=12)
ax.set_xlim([-5, None])


If coordinates are unavailable—or if you prefer not to visualize in 2D—we can instead show clustering results as a **structure plot** (stacked bar charts).

*Note: This can be slow for large datasets. If your notebook is memory-limited, use with caution.*

In [ ]:
from clumppling.utils import get_uniq_lb_sep
from clumppling.plot import plot_membership

In [ ]:
# get group separations based on truth labels
grp_lbs, grp_indices, grp_seps = get_uniq_lb_sep(truth_labels)

In [ ]:
# add ground truth to the modes to plot
modes_to_plot.insert(0, ('ground.truth', 'ground.truth'))

fig = plt.figure(figsize=(9,len(modes_to_plot)*0.7), dpi=150)
gs = fig.add_gridspec(len(modes_to_plot)+1, 1, width_ratios=[1])

for i, mode in enumerate(modes_to_plot):
    mode_full_name = f"{mode[0]}_{mode[1]}"
    ax_membership = fig.add_subplot(gs[i])
    # get aligned Q matrix for the mode
    alignedQ = comp_res.Q_by_mode[mode_full_name]
    
    # plot membership as stacked barplot (structure plot)
    plot_membership(alignedQ, default_colors, ax=ax_membership, ylab="", title="", fontsize=14)
    # use the mode full name as y label
    ax_membership.set_ylabel(mode_full_name, fontsize=10, weight='bold', rotation=0, ha='right', va='center')

    # add vertical lines to separate groups based on the cell-reorder reference  mode
    for v in grp_seps:
        ax_membership.axvline(v, ymin=-1.5, ymax=1, color='darkgray', ls='--', lw=0.3, clip_on=False)
    if i==len(modes_to_plot)-1:
        for v in grp_seps:
            ax_membership.set_xticks(grp_indices)
            ax_membership.set_xticklabels(np.arange(len(grp_lbs))+1, fontsize=9, rotation=0)
            ax_membership.tick_params(axis='x', length=0)  
    
fig.tight_layout()


We can similarly visualize the difference in cluster memberships in structure plots.

In [ ]:
# compute difference matrices against reference mode
mat_diffs = aoc.get_compmodels_diff_matrices_against_ref(
    comp_res,
    pair_mappings,
    ref_mode=ref_mode,
    strict_pair_mapping=True,
)

In [ ]:
fig = plt.figure(figsize=(9,len(modes_to_plot)), dpi=300)
gs = fig.add_gridspec(len(modes_to_plot)+1, 1, width_ratios=[1])

for i, mode in enumerate(modes_to_plot[1:]):
    mode_full_name = f"{mode[0]}_{mode[1]}"

    ax_diff = fig.add_subplot(gs[i])
    # get difference Q matrix for the mode
    diffQ = mat_diffs[mode[0]][mode[1]]
    # calculate normalized Hamming distance (NHD): for hard-clustering only
    nhd = np.sum(np.sum(diffQ, axis=1)>0.5)/diffQ.shape[0]
    # plot difference membership as stacked barplot (structure plot)
    plot_membership(diffQ, default_colors, ax=ax_diff, ylab="", title="", fontsize=14)
    # use the prepared labels instead
    if mode[0]=="kmeans":
        ax_diff.set_ylabel(mode_full_name+"\n$NHD={:.3f}$".format(nhd), fontsize=10, rotation=0, ha='right', va='center')
    else:
        ax_diff.set_ylabel(mode_full_name, fontsize=10, rotation=0, ha='right', va='center')

    # add vertical lines to separate groups based on cell-reorder reference mode
    for v in grp_seps:
        ax_diff.axvline(v, ymin=-1.5, ymax=1, color='darkgray', ls='--', lw=0.3, clip_on=False)
    if i==len(modes_to_plot[1:])-1:
        for v in grp_seps:
            ax_diff.set_xticks(grp_indices)
            ax_diff.set_xticklabels(np.arange(len(grp_lbs))+1, fontsize=11, rotation=0)
            ax_diff.tick_params(axis='x', length=0)  

fig.suptitle(f"Difference in Cluster Memberships vs. ground truth", fontsize=14)    
fig.tight_layout()

### Advanced Model Comparison
Besides aligned clustering results, we can also analyze and compare the associated feature metrics (if using mixed-membership clustering) across multiple models. 

Here we show how to compare results from model NMF and model LDA in our toy dataset.